In [1]:
import sys
sys.path.append("../../kaggle.playground.11.26")
import importlib
from pathlib import Path
from datetime import datetime

import joblib
import pandas as pd


from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from utils.training_automatisation import log_run, run_experiment



In [2]:


# Data files
data_dir = Path("/home/paul/.cache/kagglehub/competitions/playground-series-s6e11")
train_file = data_dir / "df_encoded_train_hot_ones.csv"  # Set to your encoded training file
test_file = data_dir / "df_encoded_test_hot_ones.csv"  # Set to your encoded test file

target = "is_fraud"
id_column = "id"

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)
feature_columns = [col for col in train_df.columns if col != target and col != id_column]


X_train = train_df[feature_columns]
y_train = train_df[target]

X_test = test_df[feature_columns]



In [3]:
num_cols = [c for c in X_train.select_dtypes(include="number").columns
            if X_train[c].nunique() > 2]

scaler = ColumnTransformer([("scale", StandardScaler(), num_cols)],
                           remainder="passthrough")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#model to train 

MODELS = {
    "logistic_regression_basic": {
        "active": False,
        "type": "linear",
        "model": Pipeline([
            ("preprocess", scaler),
            ("clf", LogisticRegression(class_weight="balanced",
                                       max_iter=1000, random_state=42)),
        ]),
    },
    "random_forest": {
        "active": True,
        "type": "tree",
        "model": Pipeline([
            ("preprocess", "passthrough"),
            ("clf", RandomForestClassifier(class_weight="balanced",
                                           n_jobs=-1, random_state=42)),
        ]),
        "param_grid": {
            "clf__n_estimators": [200, 400],
            "clf__max_depth": [None, 10, 20],
            "clf__min_samples_leaf": [1, 5],
        },
    },
}


In [ ]:
models_dir = data_dir / "models_weights"

to_run = {n: s for n, s in MODELS.items() if s.get("active", True)}

for name, spec in to_run.items():
    run_experiment(name, spec, X_train, y_train, X_test, test_df, id_column, models_dir, data_dir, cv=cv)